# Manga / Webtoon → 대사 추출 + 번역 (Colab)

한국어/일본어 전용 경량 버전입니다.

- 한국어 → **Pororo/BrainOCR (PyTorch CUDA)**
- 일본어 → **MangaOCR**
- 위치 검출/말풍선 grouping → **Koharu RF-DETR segmentation**
- 일본어 번역 → 필요 시 Qwen
- **PaddlePaddle / PaddleOCR / PaddleX / ONNX Runtime / PySide6 사용 안 함**


## 1. 패키지 설치

Colab의 기존 PyTorch/CUDA를 그대로 사용합니다.

이 셀은 필요한 패키지만 설치하며, 설치가 끝나도 런타임을 강제로 종료하지 않습니다.


In [ ]:
import subprocess
import sys

설치할_패키지 = [
    "rfdetr==1.7.0",
    "safetensors>=0.5",
    "huggingface_hub>=0.27",
    "manga-ocr>=0.1.11",
    "transformers>=4.51",
    "accelerate>=1.2",
    "bitsandbytes>=0.45",
    "lingua-language-detector>=2.0",
    "pymupdf>=1.24",
    "pillow",
    "numpy",
    "tqdm",
    "matplotlib",
    "mahotas>=1.4.18",
    "requests>=2.31.0",
    "certifi",
    "wget>=3.2",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *설치할_패키지,
    ],
    check=True,
)

print("[설치 완료]")
print("Paddle / ONNX / PySide6는 설치하지 않습니다.")
print("런타임을 강제로 재시작하지 않습니다.")


## 2. 저장소 가져오기


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

저장소_경로 = Path("/content/manga2text_tmp")
comic_translate_경로 = Path("/content/comic-translate")

for path in [
    저장소_경로,
    comic_translate_경로,
]:
    if path.exists():
        shutil.rmtree(path)

subprocess.run(
    [
        "git",
        "clone",
        "-q",
        "https://github.com/HisameOgasahara/manga2text_tmp.git",
        str(저장소_경로),
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "-q",
        "https://github.com/ogkalu2/comic-translate.git",
        str(comic_translate_경로),
    ],
    check=True,
)

os.environ["COMIC_TRANSLATE_PATH"] = str(comic_translate_경로)

sys.path.insert(0, str(저장소_경로))
sys.path.insert(0, str(comic_translate_경로))

print("[저장소 준비 완료]")
print("manga2text     :", 저장소_경로)
print("comic-translate:", comic_translate_경로)


## 3. 실행 환경 확인


In [ ]:
import platform

import torch

print("[실행 환경]")
print("Python    :", platform.python_version())
print("PyTorch   :", torch.__version__)
print("CUDA 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))


## 4. 사용자 설정

`글자_읽는_방법=자동`이면:

- 한국어 → Pororo/BrainOCR
- 일본어 → MangaOCR


In [ ]:
import os
from pathlib import Path

os.environ["DISABLE_MODEL_SOURCE_CHECK"] = "True"

from manga2text_pipeline import (
    build_language_detector,
    classify_inputs,
    collect_page_images,
    describe_input_mode,
    load_koharu_detector,
    load_translation_model,
    make_preview_images,
    process_pages,
    save_results,
)

from manga2text_pororo import (
    auto_detect_source_language,
    install_pororo_overrides,
    load_ocr_backend,
    resolve_ocr_configuration,
)

install_pororo_overrides()


# @title 사용자 설정

언어_자동_판별 = True  # @param {type:"boolean"}
원문_언어 = "한국어"  # @param ["한국어", "일본어"]

글자_읽는_방법 = "자동"  # @param ["자동", "PororoOCR", "MangaOCR"]
읽기_방향 = "자동"  # @param ["자동", "오른쪽→왼쪽 (일본 만화)", "왼쪽→오른쪽 (한국 만화/웹툰)"]

외국어_한국어_번역 = True  # @param {type:"boolean"}
번역_모델 = "Qwen3-1.7B (가볍고 빠름)"  # @param ["Qwen3-1.7B (가볍고 빠름)", "Qwen3-4B (품질 우선)"]

효과음도_읽기 = False  # @param {type:"boolean"}
PDF_화질_DPI = 200  # @param {type:"integer"}
처리할_페이지_수 = 0  # @param {type:"integer"}
동시에_준비할_작업_수 = 4  # @param {type:"integer"}
진단_로그_보기 = True  # @param {type:"boolean"}


언어_코드 = {
    "한국어": "ko",
    "일본어": "ja",
}

OCR_코드 = {
    "자동": "auto",
    "PororoOCR": "pororo",
    "MangaOCR": "manga",
}

읽기_방향_코드 = {
    "자동": "auto",
    "오른쪽→왼쪽 (일본 만화)": "rtl",
    "왼쪽→오른쪽 (한국 만화/웹툰)": "ltr",
}

번역_모델_코드 = {
    "Qwen3-1.7B (가볍고 빠름)": "Qwen/Qwen3-1.7B",
    "Qwen3-4B (품질 우선)": "Qwen/Qwen3-4B",
}


AUTO_DETECT_SOURCE_LANGUAGE = 언어_자동_판별
SOURCE_LANGUAGE = 언어_코드[원문_언어]
OCR_BACKEND = OCR_코드[글자_읽는_방법]
READING_DIRECTION = 읽기_방향_코드[읽기_방향]

ENABLE_TRANSLATION = 외국어_한국어_번역
TRANSLATION_MODEL = 번역_모델_코드[번역_모델]
INCLUDE_SFX = 효과음도_읽기

PDF_DPI = PDF_화질_DPI
PAGE_LIMIT = (
    None
    if 처리할_페이지_수 <= 0
    else 처리할_페이지_수
)
INPUT_WORKERS = max(
    1,
    동시에_준비할_작업_수,
)
DEBUG_LOG = 진단_로그_보기

PORORO_DEVICE = "cuda"

MAX_NEW_TOKENS = 256
CROP_PADDING = 8
ROW_TOLERANCE = 80
AUTO_LANGUAGE_SAMPLE_CROPS = 3
DEBUG_SAMPLES_PER_PAGE = 3

CLASS_THRESHOLDS = {
    0: 0.25,
    1: 0.20,
    2: 0.50,
    3: 0.50,
}


WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [
    INPUT_DIR,
    PAGE_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("[현재 설정]")
print("언어 자동 판별:", AUTO_DETECT_SOURCE_LANGUAGE)
print("글자 읽는 방법:", 글자_읽는_방법)
print("Pororo 장치    :", PORORO_DEVICE)
print("외국어 번역    :", ENABLE_TRANSLATION)


## 4-A. 읽기 순서 설정 (선택)


In [ ]:
from reading_order_ui import show_reading_order_controls

reading_order_ui = show_reading_order_controls()


## 5. 만화 업로드 + 미리보기


In [ ]:
import shutil

import matplotlib.pyplot as plt
from google.colab import files

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

uploaded_files = files.upload()

for filename, file_bytes in uploaded_files.items():
    output_path = INPUT_DIR / filename
    output_path.write_bytes(file_bytes)


input_groups = classify_inputs(INPUT_DIR)

print("[입력 확인]")
print("입력 종류 :", describe_input_mode(input_groups))
print("이미지 수 :", len(input_groups["images"]))
print("PDF 수    :", len(input_groups["pdfs"]))


preview_items = make_preview_images(
    input_dir=INPUT_DIR,
    max_items=8,
    pdf_preview_pages=3,
)

if not preview_items:
    raise RuntimeError(
        "미리보기 가능한 이미지 또는 PDF가 없습니다."
    )

cols = min(
    4,
    len(preview_items),
)
rows = (
    len(preview_items) + cols - 1
) // cols

plt.figure(
    figsize=(4 * cols, 5 * rows)
)

for index, (label, image) in enumerate(
    preview_items,
    start=1,
):
    ax = plt.subplot(
        rows,
        cols,
        index,
    )
    ax.imshow(image)
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6. 페이지 준비 + 한국어/일본어 자동 판별


In [ ]:
import shutil

if PAGE_DIR.exists():
    shutil.rmtree(PAGE_DIR)

PAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
    workers=INPUT_WORKERS,
)

print("준비된 페이지:", len(page_paths))


detector = load_koharu_detector()


if AUTO_DETECT_SOURCE_LANGUAGE:
    (
        selected_source_language,
        auto_language_details,
    ) = auto_detect_source_language(
        page_paths=page_paths,
        detector=detector,
        class_thresholds=CLASS_THRESHOLDS,
        crop_padding=CROP_PADDING,
        max_crops=AUTO_LANGUAGE_SAMPLE_CROPS,
        pororo_device=PORORO_DEVICE,
    )
else:
    selected_source_language = SOURCE_LANGUAGE


ocr_config = resolve_ocr_configuration(
    source_language=selected_source_language,
    ocr_backend=OCR_BACKEND,
    reading_direction=READING_DIRECTION,
)

print("[OCR routing]")
print(ocr_config)


## 7. OCR / 번역 모델 로드


In [ ]:
ocr_model = load_ocr_backend(
    backend=ocr_config["ocr_backend"],
    pororo_device=PORORO_DEVICE,
)

language_detector, language_to_code = (
    build_language_detector()
)


translation_tokenizer = None
translation_model = None

if (
    ENABLE_TRANSLATION
    and selected_source_language == "ja"
):
    (
        translation_tokenizer,
        translation_model,
    ) = load_translation_model(
        TRANSLATION_MODEL
    )


print("[모델 준비 완료]")
print(
    "OCR backend:",
    ocr_config["ocr_backend"],
)


## 8. 전체 페이지 처리

Koharu가 `panel / bubble / text` 영역을 검출하고 grouping한 뒤,
한국어 crop은 Pororo, 일본어 crop은 MangaOCR이 인식합니다.


In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=ocr_config["ocr_backend"],
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=ocr_config["reading_direction"],
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
    debug=DEBUG_LOG,
    debug_samples_per_page=DEBUG_SAMPLES_PER_PAGE,
)

print("records:", len(records))


## 9. 결과 저장 + 미리보기


In [ ]:
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)


print()

for record in records[:50]:
    original = record["original"].replace(
        "\n",
        " ",
    )
    korean = record["korean"].replace(
        "\n",
        " ",
    )

    print(
        f"[p.{record['page']:03d} / "
        f"{record['order']:02d}] "
        f"{record['language']} | "
        f"{original}"
        + (
            f" -> {korean}"
            if korean != original
            else ""
        )
    )


## 10. 결과 다운로드


In [ ]:
from google.colab import files

files.download(str(txt_path))
files.download(str(jsonl_path))
